# Build LSTM Models by Sector

This notebook trains one LSTM return-regression model for each sector. Each model uses a 30-day input window and predicts the 5-trading-day return after the prediction date.

The reusable dataset, model, scaling, metric, and filename helpers come from `core.lstm_helpers`.

In [2]:
pip install tensorflow scikit-learn matplotlib pandas yfinance ta

  Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 41.9 MB/s  0:00:00 eta 0:00:01
Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl (274 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.9 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import gc
import sys
from pathlib import Path

# Add the backend folder to the path so this notebook can reuse the shared project code.
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

# Use the same feature function and ticker groups as the baseline notebook.
from core.fetch_data import SELECTED_FEATURE_COLS, get_ticker_features
from core.lstm_helpers import (
    build_lstm_model,
    build_sector_data,
    inverse_scale_predictions,
    regression_metrics,
    sector_slug,
)
from core.tickers import TICKER_GROUPS, TICKERS
from tensorflow import keras

Matplotlib is building the font cache; this may take a moment.


## 1. Configuration

In [4]:
# Start the raw feature download before the model period.
# This gives the moving averages enough warm-up data before 2022.
FEATURE_START = "2021-10-01"

# Keep the train, validation, and test periods separate by date.
# This makes the test period a clean final check on 2025 data.
TRAIN_START = "2022-01-01"
TRAIN_MODEL_END = "2023-12-31"
VALIDATION_START = "2024-01-01"
VALIDATION_END = "2024-12-31"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

# Download past 2025 because a late-2025 row still needs 5 future trading days.
DOWNLOAD_END = "2026-01-10"

# Use a 30-day input window to predict the 5-trading-day return.
FORECAST_HORIZON = 5
WINDOW_SIZE = 30

# These settings control the model training loop.
MAX_EPOCHS = 30
BATCH_SIZE = 128
EARLY_STOPPING_PATIENCE = 5
RANDOM_SEED = 42

# Save the sector models to a separate folder.
MODEL_DIR = Path("../trained_models/lstm_sector_return_regression")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Use the shared feature list from fetch_data.py and remove duplicates defensively.
FEATURE_COLS = list(dict.fromkeys(SELECTED_FEATURE_COLS))
SELECTED_SECTORS = list(TICKER_GROUPS)

print(f"{len(SELECTED_SECTORS)} sector models")
print(f"{len(FEATURE_COLS)} features per day, {WINDOW_SIZE}-day sequences")
print(f"Forecast target: {FORECAST_HORIZON}-trading-day return")

9 sector models
18 features per day, 30-day sequences
Forecast target: 5-trading-day return


## 2. Load Ticker Features

In [5]:
# Calculate the features once per ticker so every model uses the same source data.
ticker_features = {}

for ticker in TICKERS:
    df = get_ticker_features(
        ticker=ticker,
        start=FEATURE_START,
        end=DOWNLOAD_END,
        smooth_outliers=True,
    )

    # Skip tickers that cannot create one complete sequence and one future target.
    required_rows = WINDOW_SIZE + FORECAST_HORIZON + 1
    if df.empty or len(df) < required_rows:
        print(f"{ticker}: skipped (only {len(df)} rows)")
        continue

    ticker_features[ticker] = df
    print(f"{ticker}: {len(df)} rows ({df.index[0].date()} to {df.index[-1].date()})")

print(f"Loaded {len(ticker_features)} of {len(TICKERS)} requested tickers.")

AAPL: 965 rows (2022-03-08 to 2026-01-09)
MSFT: 965 rows (2022-03-08 to 2026-01-09)
GOOGL: 965 rows (2022-03-08 to 2026-01-09)
GOOG: 965 rows (2022-03-08 to 2026-01-09)
AMZN: 965 rows (2022-03-08 to 2026-01-09)
META: 965 rows (2022-03-08 to 2026-01-09)
NVDA: 965 rows (2022-03-08 to 2026-01-09)
TSLA: 965 rows (2022-03-08 to 2026-01-09)
AVGO: 965 rows (2022-03-08 to 2026-01-09)
ORCL: 965 rows (2022-03-08 to 2026-01-09)
CRM: 965 rows (2022-03-08 to 2026-01-09)
ADBE: 965 rows (2022-03-08 to 2026-01-09)
AMD: 965 rows (2022-03-08 to 2026-01-09)
INTC: 965 rows (2022-03-08 to 2026-01-09)
CSCO: 965 rows (2022-03-08 to 2026-01-09)
IBM: 965 rows (2022-03-08 to 2026-01-09)
QCOM: 965 rows (2022-03-08 to 2026-01-09)
TXN: 965 rows (2022-03-08 to 2026-01-09)
NOW: 965 rows (2022-03-08 to 2026-01-09)
INTU: 965 rows (2022-03-08 to 2026-01-09)
ACN: 965 rows (2022-03-08 to 2026-01-09)
DIS: 965 rows (2022-03-08 to 2026-01-09)
NFLX: 965 rows (2022-03-08 to 2026-01-09)
CMCSA: 965 rows (2022-03-08 to 2026-01-0

## 3. Train Every Sector Model

In [6]:
# Train one model per sector so each model only sees related stocks.
by_sector_results = []
by_sector_training_histories = {}
by_sector_trained_artifacts = {}

by_sector_total_runs = len(SELECTED_SECTORS)

for run_number, sector in enumerate(SELECTED_SECTORS, start=1):
    by_sector_tickers = TICKER_GROUPS[sector]
    print(
        f"[{run_number}/{by_sector_total_runs}] {sector} - {FORECAST_HORIZON}d return regression"
    )

    by_sector_dataset = build_sector_data(
        sector_tickers=by_sector_tickers,
        ticker_features=ticker_features,
        feature_cols=FEATURE_COLS,
        train_start=TRAIN_START,
        train_model_end=TRAIN_MODEL_END,
        validation_start=VALIDATION_START,
        validation_end=VALIDATION_END,
        test_start=TEST_START,
        test_end=TEST_END,
        window_size=WINDOW_SIZE,
        forecast_horizon=FORECAST_HORIZON,
    )
    if by_sector_dataset is None:
        print("  skipped: one or more dataset splits are empty")
        continue

    # Unpack the arrays for the current sector before fitting the model.
    by_sector_X_train = by_sector_dataset["train"]["X"]
    by_sector_y_train = by_sector_dataset["train"]["y_scaled"]
    by_sector_X_validation = by_sector_dataset["validation"]["X"]
    by_sector_y_validation = by_sector_dataset["validation"]["y_scaled"]
    by_sector_X_test = by_sector_dataset["test"]["X"]
    by_sector_y_test = by_sector_dataset["test"]["y"]
    by_sector_current_close_test = by_sector_dataset["test"]["current_close"]
    by_sector_target_close_test = by_sector_dataset["test"]["target_close"]

    # Reset the model state for each sector so results do not carry over.
    keras.backend.clear_session()
    keras.utils.set_random_seed(RANDOM_SEED + run_number)

    by_sector_model = build_lstm_model((WINDOW_SIZE, len(FEATURE_COLS)))

    by_sector_early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    )

    # The validation period decides when training should stop.
    by_sector_history = by_sector_model.fit(
        by_sector_X_train,
        by_sector_y_train,
        validation_data=(by_sector_X_validation, by_sector_y_validation),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[by_sector_early_stopping],
        verbose=0,
    )

    # Convert scaled return predictions back before calculating price metrics.
    by_sector_y_pred_scaled = by_sector_model.predict(
        by_sector_X_test, batch_size=BATCH_SIZE, verbose=0
    ).flatten()
    by_sector_y_pred = inverse_scale_predictions(
        by_sector_dataset["target_scaler"], by_sector_y_pred_scaled
    )
    by_sector_naive_pred = np.zeros_like(by_sector_y_test)

    by_sector_model_metrics = regression_metrics(
        by_sector_y_pred, by_sector_current_close_test, by_sector_target_close_test
    )
    by_sector_naive_metrics = regression_metrics(
        by_sector_naive_pred, by_sector_current_close_test, by_sector_target_close_test
    )

    by_sector_model_path = (
        MODEL_DIR / f"{sector_slug(sector)}_{FORECAST_HORIZON}d_return.keras"
    )
    by_sector_model.save(by_sector_model_path)

    # Difference columns compare the LSTM with the zero-return baseline.
    by_sector_result = {
        "sector": sector,
        "forecast_horizon": f"{FORECAST_HORIZON}d",
        "tickers": len(by_sector_dataset["tickers"]),
        "train_samples": len(by_sector_dataset["train"]["y"]),
        "validation_samples": len(by_sector_dataset["validation"]["y"]),
        "test_samples": len(by_sector_y_test),
        "naive_mae": by_sector_naive_metrics["mae"],
        "mae": by_sector_model_metrics["mae"],
        "mae_difference": by_sector_naive_metrics["mae"]
        - by_sector_model_metrics["mae"],
        "naive_rmse": by_sector_naive_metrics["rmse"],
        "rmse": by_sector_model_metrics["rmse"],
        "rmse_difference": by_sector_naive_metrics["rmse"]
        - by_sector_model_metrics["rmse"],
        "naive_mape": by_sector_naive_metrics["mape"],
        "mape": by_sector_model_metrics["mape"],
        "mape_difference": by_sector_naive_metrics["mape"]
        - by_sector_model_metrics["mape"],
        "epochs": len(by_sector_history.history["loss"]),
        "model_path": str(by_sector_model_path),
    }
    by_sector_result["beats_naive"] = by_sector_result["mae_difference"] > 0

    by_sector_results.append(by_sector_result)
    by_sector_training_histories[(sector, FORECAST_HORIZON)] = by_sector_history.history

    # Keep prediction artifacts so one sector can be inspected after training.
    by_sector_trained_artifacts[(sector, FORECAST_HORIZON)] = {
        "feature_scaler": by_sector_dataset["feature_scaler"],
        "target_scaler": by_sector_dataset["target_scaler"],
        "metadata": by_sector_dataset["test"]["metadata"],
        "y_test": by_sector_y_test,
        "y_pred": by_sector_y_pred,
        "current_close": by_sector_current_close_test,
        "target_close": by_sector_target_close_test,
    }

    print(
        f"  mae={by_sector_result['mae']:.4f}, "
        f"naive_mae={by_sector_result['naive_mae']:.4f}, "
        f"mae_difference={by_sector_result['mae_difference']:.4f}, "
        f"rmse={by_sector_result['rmse']:.4f}, "
        f"mape={by_sector_result['mape']:.4f}, "
        f"epochs={by_sector_result['epochs']}"
    )

    del by_sector_model
    gc.collect()

by_sector_results_df = pd.DataFrame(by_sector_results)

by_sector_metric_cols = [
    "naive_mae",
    "mae",
    "mae_difference",
    "naive_rmse",
    "rmse",
    "rmse_difference",
    "naive_mape",
    "mape",
    "mape_difference",
]

if not by_sector_results_df.empty:
    by_sector_results_df[by_sector_metric_cols] = by_sector_results_df[
        by_sector_metric_cols
    ].round(4)
    by_sector_results_df = by_sector_results_df.sort_values(
        "mae_difference", ascending=False
    )

by_sector_results_df

[1/9] Technology and communication services - 5d return regression
  mae=10.1454, naive_mae=8.3192, mae_difference=-1.8262, rmse=15.6955, mape=0.0450, epochs=6
[2/9] Financials - 5d return regression
  mae=9.6590, naive_mae=9.2695, mae_difference=-0.3895, rmse=15.7824, mape=0.0292, epochs=6
[3/9] Healthcare - 5d return regression
  mae=9.5852, naive_mae=9.0522, mae_difference=-0.5330, rmse=19.1268, mape=0.0332, epochs=10
[4/9] Consumer staples and discretionary - 5d return regression
  mae=5.2588, naive_mae=5.1525, mae_difference=-0.1063, rmse=9.7719, mape=0.0283, epochs=6
[5/9] Industrials - 5d return regression
  mae=7.8353, naive_mae=7.7876, mae_difference=-0.0477, rmse=11.6347, mape=0.0313, epochs=6
[6/9] Energy - 5d return regression
  mae=3.5888, naive_mae=3.5033, mae_difference=-0.0855, rmse=5.0797, mape=0.0333, epochs=11
[7/9] Utilities - 5d return regression
  mae=1.7553, naive_mae=1.7189, mae_difference=-0.0363, rmse=2.4275, mape=0.0216, epochs=7
[8/9] Materials - 5d return r

,sector,forecast_horizon,tickers,train_samples,validation_samples,test_samples,naive_mae,mae,mae_difference,naive_rmse,rmse,rmse_difference,naive_mape,mape,mape_difference,epochs,model_path,beats_naive
6,Utilities,5d,6,2538,1482,1500,1.7189,1.7553,-0.0363,2.3914,2.4275,-0.0360,0.0213,0.0216,-0.0003,7,../trained_models/lstm_sector_return_regressio...,False
4,Industrials,5d,11,4653,2717,2750,7.7876,7.8353,-0.0477,11.4595,11.6347,-0.1752,0.0310,0.0313,-0.0003,6,../trained_models/lstm_sector_return_regressio...,False
5,Energy,5d,7,2961,1729,1750,3.5033,3.5888,-0.0855,5.0398,5.0797,-0.0399,0.0328,0.0333,-0.0006,11,../trained_models/lstm_sector_return_regressio...,False
3,Consumer staples and discretionary,5d,13,5499,3211,3250,5.1525,5.2588,-0.1063,9.7067,9.7719,-0.0652,0.0277,0.0283,-0.0006,6,../trained_models/lstm_sector_return_regressio...,False
8,Real estate,5d,3,1269,741,750,9.3165,9.4922,-0.1757,16.5455,16.3959,0.1496,0.0252,0.0266,-0.0014,12,../trained_models/lstm_sector_return_regressio...,False
7,Materials,5d,5,2115,1235,1250,5.6495,5.9770,-0.3275,8.4163,8.7862,-0.3698,0.0322,0.0338,-0.0016,7,../trained_models/lstm_sector_return_regressio...,False
1,Financials,5d,13,5499,3211,3250,9.2695,9.6590,-0.3895,15.2665,15.7824,-0.5160,0.0282,0.0292,-0.0010,6,../trained_models/lstm_sector_return_regressio...,False
2,Healthcare,5d,14,5922,3458,3500,9.0522,9.5852,-0.5330,18.6118,19.1268,-0.5150,0.0312,0.0332,-0.0021,10,../trained_models/lstm_sector_return_regressio...,False
0,Technology and communication services,5d,27,11421,6669,6750,8.3192,10.1454,-1.8262,13.5201,15.6955,-2.1753,0.0374,0.0450,-0.0076,6,../trained_models/lstm_sector_return_regressio...,False


## 4. Compare Sector Results

In [7]:
by_sector_comparison_cols = [
    "sector",
    "forecast_horizon",
    "tickers",
    "test_samples",
    "naive_mae",
    "mae",
    "mae_difference",
    "beats_naive",
    "naive_rmse",
    "rmse",
    "rmse_difference",
    "naive_mape",
    "mape",
    "mape_difference",
    "epochs",
    "model_path",
]

display(by_sector_results_df[by_sector_comparison_cols])

,sector,forecast_horizon,tickers,test_samples,naive_mae,mae,mae_difference,beats_naive,naive_rmse,rmse,rmse_difference,naive_mape,mape,mape_difference,epochs,model_path
6,Utilities,5d,6,1500,1.7189,1.7553,-0.0363,False,2.3914,2.4275,-0.0360,0.0213,0.0216,-0.0003,7,../trained_models/lstm_sector_return_regressio...
4,Industrials,5d,11,2750,7.7876,7.8353,-0.0477,False,11.4595,11.6347,-0.1752,0.0310,0.0313,-0.0003,6,../trained_models/lstm_sector_return_regressio...
5,Energy,5d,7,1750,3.5033,3.5888,-0.0855,False,5.0398,5.0797,-0.0399,0.0328,0.0333,-0.0006,11,../trained_models/lstm_sector_return_regressio...
3,Consumer staples and discretionary,5d,13,3250,5.1525,5.2588,-0.1063,False,9.7067,9.7719,-0.0652,0.0277,0.0283,-0.0006,6,../trained_models/lstm_sector_return_regressio...
8,Real estate,5d,3,750,9.3165,9.4922,-0.1757,False,16.5455,16.3959,0.1496,0.0252,0.0266,-0.0014,12,../trained_models/lstm_sector_return_regressio...
7,Materials,5d,5,1250,5.6495,5.9770,-0.3275,False,8.4163,8.7862,-0.3698,0.0322,0.0338,-0.0016,7,../trained_models/lstm_sector_return_regressio...
1,Financials,5d,13,3250,9.2695,9.6590,-0.3895,False,15.2665,15.7824,-0.5160,0.0282,0.0292,-0.0010,6,../trained_models/lstm_sector_return_regressio...
2,Healthcare,5d,14,3500,9.0522,9.5852,-0.5330,False,18.6118,19.1268,-0.5150,0.0312,0.0332,-0.0021,10,../trained_models/lstm_sector_return_regressio...
0,Technology and communication services,5d,27,6750,8.3192,10.1454,-1.8262,False,13.5201,15.6955,-2.1753,0.0374,0.0450,-0.0076,6,../trained_models/lstm_sector_return_regressio...
